# Recognition Audit — Per-Article Materials

Builds the `recognized_materials` column in `reports/corpus_articles.csv` straight from the
`config.py` lexicon (single source of truth, shared with `08_material_frequency`), then runs a
**precision** (false-positive) and a **recall** (missed-spelling) audit over each paper's
title + abstract. Re-running this notebook keeps the committed column in lock-step with the
lexicon, so it can never silently drift.

In [1]:
# --- Recognition build + audit setup (self-contained, config-driven) ---------
import re, sys, csv, json
from collections import Counter
from pathlib import Path
import pandas as pd

# repo root (robust to running from repo root or notebooks/)
REPO = next(base for base in [Path("."), Path("..")] if (base / "reports").exists())
sys.path.insert(0, str(REPO.resolve()))
from material_frequency.config import ROLES, RECOGNITION_EXCLUSIONS

def norm_doi(s):
    s = (s or "").strip().lower()
    for pre in ("https://doi.org/", "http://doi.org/", "https://dx.doi.org/", "doi:"):
        s = s.replace(pre, "")
    return s.strip()

def reconstruct_abstract(inv):
    if not inv:
        return ""
    pos = {}
    for w, ps in inv.items():
        for p in ps:
            pos[p] = w
    return " ".join(pos[i] for i in range(max(pos) + 1) if i in pos) if pos else ""

# --- corpus text by DOI: title and abstract kept SEPARATE (for provenance) ---
recovered = json.loads((REPO / "data" / "recovered_abstracts.json").read_text())
fields = {}
with open(REPO / "data" / "corpus.jsonl") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        d = json.loads(line)
        ab = reconstruct_abstract(d.get("abstract_inverted_index")) or recovered.get(d["id"], "")
        fields[norm_doi(d.get("doi"))] = {"title": (d.get("title") or "").lower(),
                                          "abstract": (ab or "").lower()}

# --- compile lexicon: (key, label, [(raw, compiled)]) ------------------------
# WEAK = collision-prone patterns (bare acronyms + substrings that live inside other
# materials' names). A tag supported ONLY by these is a false-positive suspect to review.
WEAK = {
    r"\balg\b", r"\bcmc\b", r"\bnacmc\b", r"\bpaa\b", r"\bcms\b", r"\bcmcs\b",
    r"\bhec\b", r"\bhpc\b", r"\blbg\b", r"\bmc\b", r"\bkgm\b", r"\bpam\b", r"\bpaam\b",
    r"\bsap\b", r"(?<!meth)acrylate", r"glucomannan",
}
materials = []  # (key, label, [(raw, compiled)])
for i, (role_label, mats) in enumerate(ROLES):
    for m in mats:
        materials.append((m["key"], m["label"], [(p, re.compile(p, re.I)) for p in m["regex"]]))

def blob(doi):
    t = fields.get(doi, {"title": "", "abstract": ""})
    return t["title"] + " " + t["abstract"]

def recognize(doi):
    """Materials whose lexicon regex hits title+abstract, minus (key,doi) exclusions."""
    text = blob(doi)
    return [label for key, label, pats in materials
            if (key, doi) not in RECOGNITION_EXCLUSIONS
            and any(rx.search(text) for _raw, rx in pats)]

# --- BUILD the column in reports/corpus_articles.csv -------------------------
src = REPO / "reports" / "corpus_articles.csv"
rows = list(csv.DictReader(open(src)))
base = [c for c in rows[0].keys() if c != "recognized_materials"]
for r in rows:
    r["recognized_materials"] = "; ".join(recognize(norm_doi(r["doi"])))
with open(src, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=base + ["recognized_materials"])
    w.writeheader(); w.writerows(rows)

# --- MECHANICAL EVIDENCE: every tag must trace to a real title/abstract hit ---
lab2pats = {}
for _k, label, pats in materials:
    lab2pats.setdefault(label, []).extend(pats)

def evidence(doi, label):
    """(field, matched_text) supporting `label` in this paper."""
    t = fields.get(doi, {"title": "", "abstract": ""})
    ev = []
    for field in ("title", "abstract"):
        for _raw, rx in lab2pats.get(label, []):
            for mt in rx.finditer(t[field]):
                ev.append((field, mt.group(0)))
    return ev

n_tags = n_ok = 0
prov = Counter()
for r in rows:
    doi = norm_doi(r["doi"])
    for label in [x.strip() for x in r["recognized_materials"].split(";") if x.strip()]:
        n_tags += 1
        ev = evidence(doi, label)
        if ev:
            n_ok += 1
            prov[ev[0][0]] += 1

# --- audit helpers used by the next two cells --------------------------------
def precision_audit():
    """Tags whose ONLY support is a collision-prone (WEAK) pattern -> FP suspects."""
    recs = []
    for r in rows:
        doi = norm_doi(r["doi"]); text = blob(doi)
        for key, label, pats in materials:
            if (key, doi) in RECOGNITION_EXCLUSIONS:
                continue
            hits = [(raw, rx) for raw, rx in pats if rx.search(text)]
            if hits and all(raw in WEAK for raw, _ in hits):
                recs.append(dict(doi=doi, material=label, via=hits[0][0],
                                 match=hits[0][1].search(text).group(0)))
    return pd.DataFrame(recs)

def recall_audit(candidates):
    """For each (material, generous_pattern): papers where it matches but the
    material is NOT tagged -> a lexicon spelling-coverage gap."""
    tags = {norm_doi(r["doi"]): {x.strip() for x in r["recognized_materials"].split(";")}
            for r in rows}
    recs = []
    for material, pat in candidates:
        rx = re.compile(pat, re.I)
        for r in rows:
            doi = norm_doi(r["doi"]); m = rx.search(blob(doi))
            if m and material not in tags[doi]:
                recs.append(dict(doi=doi, material=material, candidate=pat, match=m.group(0)))
    return pd.DataFrame(recs)

CORPUS_N = len(rows)
print(f"built column: {CORPUS_N} articles, {n_tags} tags")
print(f"evidence: {n_ok}/{n_tags} tags trace to a real hit  |  "
      f"provenance title={prov['title']} abstract={prov['abstract']}")
print(f"exclusions active: {len(RECOGNITION_EXCLUSIONS)}")

built column: 180 articles, 389 tags
evidence: 389/389 tags trace to a real hit  |  provenance title=174 abstract=215
exclusions active: 2


In [2]:
# --- Precision audit: tags resting ONLY on a collision-prone pattern ----------
# (weak-only != wrong; this is a review surface. Known real collisions are already
#  suppressed via config.RECOGNITION_EXCLUSIONS and won't appear.)
fp = precision_audit()
print(f"false-positive suspects (weak-only support): {len(fp)}")
fp

false-positive suspects (weak-only support): 23


,doi,material,via,match
0,10.1021/acsagscitech.4c00226,polyacrylic acid / polyacrylate,(?<!meth)acrylate,acrylate
1,10.1021/acsagscitech.2c00215,polyacrylic acid / polyacrylate,(?<!meth)acrylate,acrylate
2,10.1021/acsagscitech.2c00187,polyacrylic acid / polyacrylate,(?<!meth)acrylate,acrylate
3,10.1021/acsomega.5c09496,polyacrylic acid / polyacrylate,(?<!meth)acrylate,acrylate
4,10.1016/j.aaspro.2015.03.052,polyacrylic acid / polyacrylate,(?<!meth)acrylate,acrylate
5,10.2225/vol6-issue3-fulltext-6,polyacrylic acid / polyacrylate,(?<!meth)acrylate,acrylate
6,10.1016/j.proeng.2016.06.573,polyacrylamide,\bpaam\b,paam
7,10.3390/agriculture15020142,polyacrylic acid / polyacrylate,(?<!meth)acrylate,acrylate
8,10.3390/gels11120957,polyacrylic acid / polyacrylate,(?<!meth)acrylate,acrylate
9,10.3390/polym10030271,polyacrylic acid / polyacrylate,(?<!meth)acrylate,acrylate


In [3]:
# --- Recall audit: variant spellings the lexicon might miss -------------------
# After the config revision these should return ~0 rows; the list is the regression
# guard -- add new suspected variants here to re-check coverage.
CANDIDATES = [
    ("polyacrylic acid / polyacrylate", r"(?<!meth)acrylic[\s-]?acid"),
    ("polyacrylamide",                  r"\bpaam\b"),
    ("hydroxyethyl cellulose",          r"hydroxyethyl[\s-]?cellulose"),
    ("hydroxypropyl cellulose",         r"hydroxypropyl[\s-]?cellulose"),
]
gaps = recall_audit(CANDIDATES)
print(f"recall gaps (present but untagged): {len(gaps)}")
gaps

recall gaps (present but untagged): 0


""
